# 05 · A ReAct tool-calling agent

After this notebook you can build a ReAct agent by hand in LangGraph (model node, `ToolNode`, `tools_condition`, the loop edge), explain every message in its trace, and then do the same with the prebuilt agent, adding streaming, iteration limits, tool-error handling and memory.

**Time:** ~30 min · **Runs:** offline on CPU in < 40 s (about 16 short LLM calls) · **Needs:** [02 · Conditional edges and loops](02_conditional_edges_and_loops.ipynb), [function calling](../08_llm_apps/04_function_calling.ipynb)

## Why this matters in interviews

- "Walk me through the agent loop" is asked in almost every agent interview. The expected answer is concrete: the model emits a tool call, your code runs it, the result goes back as a tool message, repeat until the model answers without a tool call, with a step cap.
- "Would you use a framework or write it yourself?" You can only answer well if you know what the prebuilt agent actually is: a two-node graph you can write in fifteen lines.
- Follow-ups probe the failure modes: tools that raise, agents that never stop, contexts that grow every round.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag02` | What is the core agent loop? |
| `ag08` | What is ReAct? |
| `pr21` | What is the ReAct prompt format and why does it work? |
| `ag03` | Are function calling and tool calling the same thing? |
| `ag19` | How do you handle a tool that fails? |
| `ag16` | How do you stop an agent looping forever? |
| `ag37` | What is a scratchpad and how do you keep it from exploding? |

## 1. Setup

Offline, the stand-in model picks the tool whose name and description best match the user's message, fills the arguments from the message, and after seeing tool results answers "Based on the tool results: ...". So every offline run is one tool round and then an answer. A real model reasons more, but the messages and the loop are identical.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import ast
import operator
import warnings
from typing import Annotated, Literal

import matplotlib.pyplot as plt
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from llm_setup import langchain_chat_model

llm = langchain_chat_model(temperature=0)
print("chat model:", MODEL, "| provider:", PROVIDER)

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_graph(app, title=None, xray=False):
    """Draw a compiled LangGraph left to right: solid = fixed edge, dashed = conditional edge,
    arcs above = skip ahead, arcs below = loop back. Red text = the path-map label."""
    g = app.get_graph(xray=xray)
    edges = [(e.source, e.target, e.conditional, e.data) for e in g.edges]
    col, frontier = {"__start__": 0}, ["__start__"]
    while frontier:                                     # column = hops from START (BFS)
        nxt = []
        for n in frontier:
            for s, t, _, _ in edges:
                if s == n and t not in col and t != "__end__":
                    col[t] = col[n] + 1; nxt.append(t)
        frontier = nxt
    for n in g.nodes:
        col.setdefault(n, max(col.values()) + 1)
    col["__end__"] = max(c for n, c in col.items() if n != "__end__") + 1
    rows = {}
    for n in g.nodes:
        rows.setdefault(col[n], []).append(n)
    pos = {n: (c * 2.4, (len(ns) - 1) / 2 - i) for c, ns in rows.items() for i, n in enumerate(ns)}
    nrow = max(len(v) for v in rows.values())
    fig, ax = plt.subplots(figsize=(min(11, 2.0 * len(rows) + 0.6), min(4.5, 0.9 * nrow + 1.4)))
    boxes = {}
    for n, (x, y) in pos.items():
        parts = [{"__start__": "START", "__end__": "END"}.get(p, p) for p in n.split(":")]
        fc = {"START": "#d3f9d8", "END": "#ffe3e3"}.get(parts[-1], "#e7f0ff")
        boxes[n] = FancyBboxPatch((x - 0.8, y - 0.28), 1.6, 0.56, boxstyle="round,pad=0.05", fc=fc, ec="#3b5bdb")
        ax.add_patch(boxes[n])
        ax.text(x, y, "\n".join(parts), ha="center", va="center", fontsize=8.5 if len(parts) == 1 else 7.5)
    pairs = {(s, t) for s, t, _, _ in edges}
    for s, t, cond, data in edges:
        (x1, y1), (x2, y2) = pos[s], pos[t]
        span = col[t] - col[s]
        if (t, s) in pairs and abs(span) == 1:
            rad = 0.2                                   # two-way pair: split the arrows apart
        elif span == 1 or (span == 0 and s != t):
            rad = 0.0
        else:
            rad = -0.3 if span > 1 else -0.45           # skip ahead above, loop back below
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), patchA=boxes[s], patchB=boxes[t], arrowstyle="-|>",
                                     mutation_scale=12, lw=1.3, color="#495057", linestyle="--" if cond else "-",
                                     connectionstyle=f"arc3,rad={rad}"))
        if data is not None and str(data) != t:
            mx, my = (x1 + x2) / 2 + 0.5 * rad * (y2 - y1), (y1 + y2) / 2 - 0.5 * rad * (x2 - x1)
            ax.text(mx, my + 0.1, str(data), ha="center", fontsize=7.5, color="#c92a2a")
    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 1.0, max(xs) + 1.0); ax.set_ylim(min(ys) - 0.75, max(ys) + 0.85)
    ax.axis("off"); ax.set_title(title or "graph", fontsize=10)
    plt.tight_layout(); plt.show()

## 2. ReAct in one paragraph

**In plain English:** ReAct = *Reason + Act*. The model looks at the task, decides on an action (call a tool with some arguments), reads what the tool returned (the *observation*), and decides again, until it can answer. The original paper did this in plain text: the model wrote `Thought: ... Action: search[...]`, and the program stopped generation at `Observation:` so the model could not invent the tool's result. Modern APIs do the same thing with **native tool calling**: the model returns a structured `tool_calls` list instead of text, which is far easier to parse. "Function calling" and "tool calling" are two names for this one mechanism.

```
          ┌───────────── no tool calls: the answer ──────────────► END
 START ─► agent (LLM with tools bound) ── tool calls ─► tools (run them, return ToolMessages) ─┐
            ▲                                                                                   │
            └───────────────────────────── observations ────────────────────────────────────────┘
```

## 3. Three tools

A tool is a Python function plus a schema the model can read. `@tool` builds the schema from the signature, type hints and docstring, and that docstring is the model's only manual, so write it for the model.

In [ ]:
ORDERS = {"1042": {"status": "shipped", "eta": "2026-10-02", "total": 59.90},
          "1043": {"status": "processing", "eta": "2026-10-06", "total": 120.00}}
WEATHER = {"Paris": "18C, cloudy", "London": "14C, light rain", "Lisbon": "24C, sunny"}

@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order by its id and return status, delivery date (eta) and total."""
    if order_id not in ORDERS:
        raise ValueError(f"no order with id {order_id}")
    return ORDERS[order_id]

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return WEATHER.get(city, "no data for that city")

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression such as 12 * 30 + 5. Supports + - * / and parentheses."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        raise ValueError("only numbers and + - * / are allowed")
    return str(ev(ast.parse(expression, mode="eval").body))      # no eval(): model output is untrusted

tools = [lookup_order, get_weather, calculator]
print(lookup_order.name, "->", lookup_order.args)
print(calculator.invoke({"expression": "12 * 30 + 5"}))

`bind_tools` attaches the schemas to every request. The model does not run anything: it returns a **request** to run a tool, and your code decides whether to honour it. That boundary is where permission checks and approvals live (notebook 04).

In [ ]:
llm_with_tools = llm.bind_tools(tools)
reply = llm_with_tools.invoke([HumanMessage("What is the status of order 1042?")])
print("content   :", repr(reply.content))
print("tool_calls:", [(c["name"], c["args"]) for c in reply.tool_calls])
assert reply.tool_calls and reply.tool_calls[0]["name"] == "lookup_order"

## 4. Build the agent by hand

Two nodes and three edges:

- `agent`: calls the model with the whole message history and appends its reply.
- `tools`: the prebuilt `ToolNode` runs every tool call in the last AI message (in parallel if there are several) and appends one `ToolMessage` per call, linked by `tool_call_id`.
- `tools_condition`: the routing function: go to `"tools"` if the last message has tool calls, otherwise to `END`.
- `tools -> agent`: the loop edge that feeds observations back.

In [ ]:
def agent(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)       # -> "tools" or END
builder.add_edge("tools", "agent")                            # the loop
react = builder.compile()
draw_graph(react, "Hand-built ReAct agent: agent -> tools -> agent until no tool calls")

In [ ]:
def print_trace(messages):
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            body = "calls " + ", ".join(f"{c['name']}({c['args']})" for c in m.tool_calls)
        elif isinstance(m, ToolMessage):
            body = f"[{m.name}] {m.content}"
        else:
            body = m.content
        print(f"{type(m).__name__:13s} {body[:110]}")

out = react.invoke({"messages": [HumanMessage("What is the status of order 1042?")]})
print_trace(out["messages"])
assert [m.type for m in out["messages"]] == ["human", "ai", "tool", "ai"]
assert "shipped" in out["messages"][2].content and not out["messages"][-1].tool_calls

Read the trace as the loop: **human** asks, **ai** requests `lookup_order`, **tool** returns the real data, **ai** answers with no tool calls, so `tools_condition` routes to `END`. Two model calls for one tool round. Every extra round costs one more model call, carrying the whole history again.

When the model asks for several independent things, it can return several tool calls in one message, and `ToolNode` runs them concurrently:

In [ ]:
out = react.invoke({"messages": [HumanMessage("What's the weather in Paris and London?")]})
print_trace(out["messages"])
tool_msgs = [m for m in out["messages"] if isinstance(m, ToolMessage)]
assert len(tool_msgs) == 2 and all(m.tool_call_id for m in tool_msgs)

Each `ToolMessage` carries the `tool_call_id` of the call it answers. The API rejects a history in which a tool call has no answer, which is why every path through your graph (errors, rejections, budget exhaustion) must still produce one.

## 5. The prebuilt agent, and how it compares

LangGraph ships this exact graph as `create_react_agent`. In LangGraph 1.x it is **deprecated** in favour of LangChain's `create_agent`, which builds the same loop on LangGraph and adds *middleware* (hooks for limits, summarisation, human approval). Both still work:

In [ ]:
from langchain.agents import create_agent

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    from langgraph.prebuilt import create_react_agent
    prebuilt = create_react_agent(llm, tools)
print("warning:", str(caught[0].message)[:118] if caught else "none")

modern = create_agent(llm, tools)
question = {"messages": [HumanMessage("What is the status of order 1042?")]}
for name, app in [("hand-built", react), ("create_react_agent", prebuilt), ("create_agent", modern)]:
    msgs = app.invoke(question)["messages"]
    print(f"{name:19s} nodes={[n for n in app.get_graph().nodes if not n.startswith('__')]} "
          f"trace={[m.type for m in msgs]}")
    assert [m.type for m in msgs] == ["human", "ai", "tool", "ai"]

Same trace, same number of model calls. The prebuilt versions add options you would otherwise write yourself: a system `prompt`, `response_format` for a structured final answer, a `checkpointer`, `interrupt_before`, pre/post-model hooks, and (in `create_agent`) middleware. Rule of thumb: use the prebuilt agent until you need control flow it does not offer, then drop down to the hand-built graph, which you now know is short.

## 6. Streaming the steps

`stream_mode="updates"` yields one event per node, which is how an agent UI shows "calling lookup_order...". `stream_mode="messages"` yields LLM tokens as they are generated, tagged with the node they came from.

In [ ]:
for event in react.stream({"messages": [HumanMessage("What is the status of order 1043?")]}, stream_mode="updates"):
    for node, update in event.items():
        m = update["messages"][-1]
        what = f"tool calls {[c['name'] for c in m.tool_calls]}" if getattr(m, "tool_calls", None) else m.content[:70]
        print(f"[{node}] {what}")

In [ ]:
tokens = []
for chunk, meta in react.stream({"messages": [HumanMessage("What is the status of order 1043?")]},
                                stream_mode="messages"):
    if meta["langgraph_node"] == "agent" and chunk.content:
        tokens.append(chunk.content)
print(f"{len(tokens)} text chunks streamed from the agent node; first five: {tokens[:5]}")
print("joined:", "".join(tokens))
assert "processing" in "".join(tokens)

## 7. Limiting iterations

The offline model always stops after one round, so to watch the guards work we swap in a **stubborn model**: a plain function that returns a tool call every time, like a real model stuck re-checking the same thing. Guard 1 is the `recursion_limit` backstop from notebook 02:

In [ ]:
calls = {"model": 0}

def stubborn_agent(state: MessagesState) -> dict:
    calls["model"] += 1
    return {"messages": [AIMessage("", tool_calls=[{"name": "lookup_order", "args": {"order_id": "1042"},
                                                    "id": f"call_{calls['model']}", "type": "tool_call"}])]}

sb = StateGraph(MessagesState)
sb.add_node("agent", stubborn_agent); sb.add_node("tools", ToolNode(tools))
sb.add_edge(START, "agent"); sb.add_conditional_edges("agent", tools_condition); sb.add_edge("tools", "agent")
try:
    sb.compile().invoke({"messages": [HumanMessage("status of 1042?")]}, {"recursion_limit": 12})
except GraphRecursionError as e:
    print("GraphRecursionError after", calls["model"], "model calls:", str(e).splitlines()[0][:60])

A crash is a poor user experience. Guard 2 is an explicit **tool-round budget** in state, with a named give-up path. Note the detail that bites people: the give-up node must first answer the pending tool calls (the API rejects unanswered ones), then say what it found.

In [ ]:
MAX_ROUNDS = 3

class BudgetState(MessagesState):
    rounds: Annotated[int, operator.add]

def counted_agent(state: BudgetState) -> dict:
    return {**stubborn_agent(state), "rounds": 1}

def route(state: BudgetState) -> Literal["tools", "give_up", "__end__"]:
    if not state["messages"][-1].tool_calls:
        return END
    return "give_up" if state["rounds"] >= MAX_ROUNDS else "tools"

def give_up(state: BudgetState) -> dict:
    pending = [ToolMessage("skipped: step budget exhausted", tool_call_id=c["id"], name=c["name"])
               for c in state["messages"][-1].tool_calls]
    seen = [m.content for m in state["messages"] if isinstance(m, ToolMessage)]
    return {"messages": pending + [AIMessage(f"I could not finish (stopped after {state['rounds']} tool rounds). "
                                              f"What I found so far: {seen[-1] if seen else 'nothing'}")]}

gb = StateGraph(BudgetState)
gb.add_node("agent", counted_agent); gb.add_node("tools", ToolNode(tools)); gb.add_node("give_up", give_up)
gb.add_edge(START, "agent"); gb.add_conditional_edges("agent", route); gb.add_edge("tools", "agent")
gb.add_edge("give_up", END)
guarded = gb.compile()
out = guarded.invoke({"messages": [HumanMessage("status of 1042?")], "rounds": 0})
print(out["messages"][-1].content)
print("rounds used:", out["rounds"])
assert out["rounds"] == MAX_ROUNDS and out["messages"][-1].content.startswith("I could not finish")

The prebuilt agents have both guards built in: `create_react_agent` tracks `remaining_steps` and, when fewer than two steps remain but the model still wants a tool, answers "Sorry, need more steps to process this request." instead of raising (so pass an explicit `recursion_limit`: the 1.2 default is 10,007), and `create_agent` takes `ModelCallLimitMiddleware(run_limit=..., exit_behavior="end")` and `ToolCallLimitMiddleware`. A limit of one model call on our real agent:

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware

limited = create_agent(llm, tools, middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")])
msgs = limited.invoke({"messages": [HumanMessage("What is the status of order 1042?")]})["messages"]
print_trace(msgs)
assert "limit" in msgs[-1].content.lower()

### Why rounds are expensive: the scratchpad grows

Every round re-sends the whole history, including every tool result so far. With a tool that returns a sizeable record, the tokens *per call* grow linearly and the total tokens for the run grow roughly quadratically. We measure it on the stubborn loop, approximating tokens as characters / 4.

In [ ]:
ORDERS["1042"]["history"] = [f"2026-09-{d:02d}: scanned at depot {d}" for d in range(1, 21)]   # a chatty record
context_tokens = []

def measuring_agent(state: BudgetState) -> dict:
    sent = sum(len(str(m.content)) + len(str(getattr(m, "tool_calls", ""))) for m in state["messages"])
    context_tokens.append(sent // 4)
    return counted_agent(state)

mb = StateGraph(BudgetState)
mb.add_node("agent", measuring_agent); mb.add_node("tools", ToolNode(tools)); mb.add_node("give_up", give_up)
mb.add_edge(START, "agent"); mb.add_conditional_edges("agent", route); mb.add_edge("tools", "agent")
mb.add_edge("give_up", END)
MAX_ROUNDS = 10
mb.compile().invoke({"messages": [HumanMessage("status of 1042?")], "rounds": 0}, {"recursion_limit": 40})
cumulative = [sum(context_tokens[: i + 1]) for i in range(len(context_tokens))]
print("tokens sent per model call:", context_tokens)
print("total tokens after 10 calls:", cumulative[-1])
assert context_tokens[-1] > 5 * context_tokens[1]

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(range(1, len(context_tokens) + 1), context_tokens, color="#74c0fc", label="tokens sent in this call")
ax.plot(range(1, len(cumulative) + 1), cumulative, "o-", color="#c92a2a", label="cumulative tokens for the run")
ax.set_xlabel("model call (tool round)"); ax.set_ylabel("approx. tokens")
ax.set_title("Each round re-sends every earlier tool result"); ax.legend(); plt.tight_layout(); plt.show()
del ORDERS["1042"]["history"]; MAX_ROUNDS = 3

This is the "scratchpad explosion": the fix is to **shape observations at the tool boundary** (return the three fields the model needs, not the whole record), trim or summarise old tool results, and keep large artifacts out of the context by reference.

## 8. When a tool fails

A tool that raises is normal (bad id, timeout, 404). What happens next depends on `ToolNode(handle_tool_errors=...)`. **The default changed** in recent versions: it only converts *argument validation* errors into a message for the model, and re-raises everything else, which crashes the run:

In [ ]:
try:
    react.invoke({"messages": [HumanMessage("What is the status of order 9999?")]})
except ValueError as e:
    print("default ToolNode -> the run crashed:", e)

Choose the behaviour explicitly. `True` sends a generic error message back to the model; a string replaces it with your own text; a function lets you write an error the model can act on ("a good error is a prompt"):

In [ ]:
def humanise(e: Exception) -> str:
    if isinstance(e, ValueError) and "no order" in str(e):
        return f"{e}. Ask the customer to check the order id (it is 4 digits, e.g. 1042). Do not retry."
    return f"The tool failed ({type(e).__name__}). Try once more later, then apologise."

for label, flag in [("True", True), ("custom function", humanise)]:
    eb = StateGraph(MessagesState)
    eb.add_node("agent", agent); eb.add_node("tools", ToolNode(tools, handle_tool_errors=flag))
    eb.add_edge(START, "agent"); eb.add_conditional_edges("agent", tools_condition); eb.add_edge("tools", "agent")
    msgs = eb.compile().invoke({"messages": [HumanMessage("What is the status of order 9999?")]})["messages"]
    err = next(m for m in msgs if isinstance(m, ToolMessage))
    print(f"{label:16s} status={err.status!r} content={err.content[:90]!r}")
    assert err.status == "error" and msgs[-1].type == "ai" and not msgs[-1].tool_calls

The run survives and the model sees the error as its observation. Offline the stand-in just quotes it; a real model reads "ask the customer to check the id" and does that. Beyond this: classify errors as retryable (timeouts, 5xx: retry once) or terminal (bad arguments, 404: never retry the identical call), cap retries per tool, and never put stack traces in the context.

## 9. Memory: add a checkpointer

A ReAct agent is a graph, so memory is the same one-liner as notebook 03: compile (or create) it with a checkpointer and pass a `thread_id`.

In [ ]:
memory_agent = create_agent(llm, tools, checkpointer=InMemorySaver())
t1 = {"configurable": {"thread_id": "customer-1"}}
first = memory_agent.invoke({"messages": [HumanMessage("Hi, my name is Mamitha. Where is order 1042?")]}, t1)
print("turn 1:", first["messages"][-1].content)
second = memory_agent.invoke({"messages": [HumanMessage("What is my name?")]}, t1)
print("turn 2:", second["messages"][-1].content)
other = memory_agent.invoke({"messages": [HumanMessage("What is my name?")]}, {"configurable": {"thread_id": "customer-2"}})
print("other thread:", other["messages"][-1].content)
assert "Mamitha" in second["messages"][-1].content and "Mamitha" not in other["messages"][-1].content

## Try it yourself

**1.** Add a fourth tool, `convert_currency(amount: float, to: Literal["EUR", "GBP"])` with a fixed rate table (EUR 0.92, GBP 0.79; the enum type tells the model exactly which values are legal), bind it, and ask "Convert 59.90 dollars to EUR". Check the trace calls your tool.

In [ ]:
# Solution — try it yourself first, then run this
RATES = {"EUR": 0.92, "GBP": 0.79}

@tool
def convert_currency(amount: float, to: Literal["EUR", "GBP"]) -> str:
    """Convert an amount of US dollars to another currency: EUR or GBP."""
    return f"{amount * RATES[to.upper()]:.2f} {to.upper()}"

conv_llm = llm.bind_tools(tools + [convert_currency])
cb = StateGraph(MessagesState)
cb.add_node("agent", lambda s: {"messages": [conv_llm.invoke(s["messages"])]})
cb.add_node("tools", ToolNode(tools + [convert_currency]))
cb.add_edge(START, "agent"); cb.add_conditional_edges("agent", tools_condition); cb.add_edge("tools", "agent")
msgs = cb.compile().invoke({"messages": [HumanMessage("Convert 59.90 dollars to EUR")]})["messages"]
print_trace(msgs)
assert any(isinstance(m, ToolMessage) and m.name == "convert_currency" for m in msgs)

**2.** Repeat-call detection: in the budgeted graph, route to `give_up` as soon as the model requests **exactly the same call (name and arguments) twice**, instead of waiting for the round limit. With the stubborn model it should stop after 2 rounds.

In [ ]:
# Solution — try it yourself first, then run this
def route_no_repeats(state: BudgetState) -> Literal["tools", "give_up", "__end__"]:
    last = state["messages"][-1]
    if not last.tool_calls:
        return END
    key = lambda c: (c["name"], str(sorted(c["args"].items())))
    earlier = {key(c) for m in state["messages"][:-1] if isinstance(m, AIMessage) for c in m.tool_calls}
    if any(key(c) in earlier for c in last.tool_calls) or state["rounds"] >= MAX_ROUNDS:
        return "give_up"
    return "tools"

rb = StateGraph(BudgetState)
rb.add_node("agent", counted_agent); rb.add_node("tools", ToolNode(tools)); rb.add_node("give_up", give_up)
rb.add_edge(START, "agent"); rb.add_conditional_edges("agent", route_no_repeats); rb.add_edge("tools", "agent")
rb.add_edge("give_up", END)
out = rb.compile().invoke({"messages": [HumanMessage("status of 1042?")], "rounds": 0})
print("rounds:", out["rounds"], "|", out["messages"][-1].content[:60])
assert out["rounds"] == 2

**3.** Observation shaping: write `lookup_order_brief(order_id)` that returns only `status` and `eta` as one short sentence, and compare the length of its `ToolMessage` with `lookup_order`'s for order 1043.

In [ ]:
# Solution — try it yourself first, then run this
@tool
def lookup_order_brief(order_id: str) -> str:
    """Look up an order by its id and return its status and delivery date."""
    o = ORDERS[order_id]
    return f"Order {order_id} is {o['status']}, expected {o['eta']}."

full = str(lookup_order.invoke({"order_id": "1043"}))
brief = lookup_order_brief.invoke({"order_id": "1043"})
print(f"full ({len(full)} chars): {full}\nbrief ({len(brief)} chars): {brief}")
assert len(brief) < len(full)

## Say it in an interview

**30-second answer (the agent loop):** "The model gets the conversation plus tool schemas. If it returns tool calls, my code validates and executes them and appends one tool message per call, linked by id; then the model is called again with the observations. It stops when the model answers without a tool call, or when a guard fires. In LangGraph that is two nodes, `agent` and `tools`, a conditional edge with `tools_condition`, and a loop edge back. The guards are a step budget with a graceful give-up, repeat-call detection, a cost ceiling, and `recursion_limit` as the backstop."

**30-second answer (ReAct):** "Reason, act, observe, repeat. The original text format needed a stop sequence at 'Observation:' so the model couldn't hallucinate tool results; native tool calling enforces that structurally, because only my code can write a tool message."

**Facts to have ready:**
- `llm.bind_tools(tools)`, `ToolNode(tools)`, `tools_condition`; the model *requests*, your code *executes*.
- `ToolNode` runs parallel calls concurrently; every call needs a `ToolMessage` with its `tool_call_id`.
- In LangGraph 1.x `create_react_agent` is deprecated for `langchain.agents.create_agent` (same loop, plus middleware such as `ModelCallLimitMiddleware` and `HumanInTheLoopMiddleware`).
- `ToolNode`'s default error handling re-raises non-validation errors; set `handle_tool_errors` deliberately.
- Cost per run ≈ (tool rounds + 1) model calls, each re-sending a growing history.

**Follow-ups they ask:** *"Framework or hand-rolled?"* The loop is short; I'd hand-roll for full transparency or use the prebuilt for speed, and move to a custom graph when I need approvals, custom routing or checkpoints. *"How do you know the agent is done?"* Prefer a programmatic post-condition or a typed `finish` tool over "no tool call in the reply".

**Traps:** unanswered tool calls after an error or a give-up (the next API call fails), `eval()` on model-supplied expressions, giving the model the full raw tool payload, and a step cap with no give-up message.

## Next

- [06 · Multi-agent supervisor](06_multi_agent_supervisor.ipynb): several of these agents, coordinated.
- [Agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) · [tool design and function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb) · [LangChain agents and memory](../13_langchain/06_agents_and_memory.ipynb) · [agent evaluation](../12_evaluation/04_agent_evaluation.ipynb)
- Theory: [LangGraph course](../../langgraph/index.html) (chapters 6–7, 11), [agentic AI flow](../../agentic_ai_flow/index.html), [interview bank](../../ai_interview_prep/index.html).